# H0 · Shared foundations for hybrid mission planning

Run the cells from top to bottom with **Lunar H0 (Python 3.12)**. This notebook uses the same `shared.hybrid` models as the application. It performs **no quantum execution, optimization, trajectory propagation or safety certification**. H1 adds a classical reference; H2 adds QUBO/QAOA.

All five supplied scenarios are synthetic teaching inputs. Three missions have four options each. Visibility, station capacity and vehicle physics have not been validated for these examples.

In [1]:
from pathlib import Path
import os
import sys
import json
from html import escape
from tempfile import TemporaryDirectory
from IPython.display import HTML, display
from pydantic import ValidationError

start = Path(os.environ.get("LUNAR_REPO_ROOT", Path.cwd())).resolve()
candidates = [start, *start.parents, start / "h0-hybrid-workspace"]
ROOT = next((p for p in candidates if (p / "shared/hybrid.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository, or set LUNAR_REPO_ROOT to your checkout.")
sys.path.insert(0, str(ROOT))
from shared.hybrid import (
    Scenario, StateSnapshot, Plan, load_scenario, save_scenario,
    canonical_json, fingerprint, validate_plan_references,
)
import shared.hybrid
assert Path(shared.hybrid.__file__).resolve() == (ROOT / "shared/hybrid.py").resolve()
print("Python:", sys.version.split()[0])
print("Repository:", ROOT)
print("Schema version: 1 — no solver executed")

Python: 3.12.4
Repository: /Users/tk/.codex/worktrees/task1-ingestion-cache/Nasa-Space-Apps-Challenge
Schema version: 1 — no solver executed


## 1. Load the benchmark cases

A structurally valid document can describe an infeasible or incomplete problem. In particular, **zero capacity**, **unknown capacity**, and **unlimited capacity** are not interchangeable. Unlimited capacity is not supported implicitly.

In [2]:
examples = ROOT / "shared/examples/hybrid"
scenarios = {path.stem: load_scenario(path) for path in sorted(examples.glob("*.json"))}
for name, scenario in scenarios.items():
    print(f"{name:18} | missions={len(scenario.missions)} | options={len(scenario.candidates)} | capacity={scenario.resources[0].capacity}")
assert len(scenarios) == 5

independent        | missions=3 | options=12 | capacity=3.0
infeasible         | missions=3 | options=12 | capacity=0.0
missing-capacity   | missions=3 | options=12 | capacity=None
resource-outage    | missions=3 | options=12 | capacity=1.0
shared-conflict    | missions=3 | options=12 | capacity=1.0


## 2. Inspect the common decision problem

Every mission requires exactly one selected option, completed by its deadline. Resource intervals use **[start, end)**: adjacent bookings do not overlap. Channel demand is concurrent; joule demand is a cumulative budget. A future solver must not infer a link budget, charging model or physical reachability from these declarations.

In [3]:
scenario = scenarios["shared-conflict"]
rows = []
for option in scenario.candidates:
    rows.append("<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in (
        option.option_id, option.mission_id, option.window.start_utc.isoformat(),
        option.window.end_utc.isoformat(), option.metrics["science_value"],
        option.demands[0].amount,
    )) + "</tr>")
display(HTML("<table><thead><tr><th>Option</th><th>Mission</th><th>Start UTC</th><th>End UTC</th><th>Synthetic science score</th><th>Channels</th></tr></thead><tbody>" + "".join(rows) + "</tbody></table>"))
print("Independent choices:", 4 ** 3, "combinations before constraints; H1 will evaluate them.")

Option,Mission,Start UTC,End UTC,Synthetic science score,Channels
mission-1-option-1,mission-1,2026-11-04T00:00:00+00:00,2026-11-04T01:00:00+00:00,10.0,1.0
mission-1-option-2,mission-1,2026-11-04T01:00:00+00:00,2026-11-04T02:00:00+00:00,9.0,1.0
mission-1-option-3,mission-1,2026-11-04T02:00:00+00:00,2026-11-04T03:00:00+00:00,7.0,1.0
mission-1-option-4,mission-1,2026-11-04T03:00:00+00:00,2026-11-04T04:00:00+00:00,6.0,1.0
mission-2-option-1,mission-2,2026-11-04T00:00:00+00:00,2026-11-04T01:00:00+00:00,10.0,1.0
mission-2-option-2,mission-2,2026-11-04T01:00:00+00:00,2026-11-04T02:00:00+00:00,9.0,1.0
mission-2-option-3,mission-2,2026-11-04T02:00:00+00:00,2026-11-04T03:00:00+00:00,7.0,1.0
mission-2-option-4,mission-2,2026-11-04T03:00:00+00:00,2026-11-04T04:00:00+00:00,6.0,1.0
mission-3-option-1,mission-3,2026-11-04T00:00:00+00:00,2026-11-04T01:00:00+00:00,10.0,1.0
mission-3-option-2,mission-3,2026-11-04T01:00:00+00:00,2026-11-04T02:00:00+00:00,9.0,1.0


Independent choices: 64 combinations before constraints; H1 will evaluate them.


## 3. Export, reload and identify exactly the same input

The fingerprint includes scenario revision, state, events, sources, objectives and constraints. Collection order, UTC `Z` versus `+00:00`, and integer/float spelling do not change it. Coordinate component order and changed values do. V1 is a Python canonicalization contract, not a claim of compliance with an external canonical JSON standard.

In [4]:
with TemporaryDirectory() as folder:
    exported = Path(folder) / "scenario.json"
    save_scenario(scenario, exported)
    reloaded = load_scenario(exported)
    assert reloaded == scenario
    assert fingerprint(reloaded) == fingerprint(scenario)
print("Input SHA-256:", fingerprint(scenario))
print("Round trip passed; temporary export removed.")

Input SHA-256: dd410a8ff30a5e16d701a87f19a6c3f5d3ff5597efc43821606615c3e2f0284d
Round trip passed; temporary export removed.


## 4. Make an explicit, validated change

Edit the raw document, increment its revision, then validate it again. Do not mutate nested model dictionaries in place. The export/hash functions revalidate to catch invalid nested edits.

In [5]:
changed_data = scenario.model_dump(mode="json")
changed_data["revision"] += 1
changed_data["resources"][0]["capacity"] = 2
changed = Scenario.model_validate(changed_data)
assert fingerprint(changed) != fingerprint(scenario)
print("Original capacity:", scenario.resources[0].capacity)
print("Modified capacity:", changed.resources[0].capacity)
print("New fingerprint:", fingerprint(changed))

Original capacity: 1.0
Modified capacity: 2.0
New fingerprint: 154f9873dab9e24f810be5965aac94d442bfdb3af6e2da3bbcfddf5e4aa3eb89


## 5. Invalid input fails explicitly; unknown remains unknown

A negative capacity is malformed input. Zero capacity is valid data, even when it makes every plan infeasible. Unknown capacity is `None`, and requires an explicit policy before H1/H2 can claim feasibility.

In [6]:
bad = scenario.model_dump(mode="json")
bad["resources"][0]["capacity"] = -1
try:
    Scenario.model_validate(bad)
except ValidationError as error:
    print("Expected validation error:", error.errors()[0]["msg"])
else:
    raise AssertionError("Negative capacity was incorrectly accepted")
assert scenarios["infeasible"].resources[0].capacity == 0
assert scenarios["missing-capacity"].resources[0].capacity is None
print("Zero and unknown are distinct.")

Expected validation error: Input should be greater than or equal to 0
Zero and unknown are distinct.


## 6. Keep simulated truth, observations and estimates separate

Below are explicitly synthetic examples in Moon-centered, rotating **MOON_ME** coordinates. Velocity means the derivative of those rotating-frame coordinates. No frame transformation or physical trajectory is computed. Standard deviations express uncertainty; H0 does not propagate it.

In [7]:
base = scenarios["resource-outage"].states[0].model_dump(mode="json")
truth = StateSnapshot.model_validate(base | {"state_id": "truth-demo", "record_kind": "ground_truth", "propellant_kg": 50})
observation = StateSnapshot.model_validate(base | {
    "state_id": "sensor-demo", "record_kind": "observation", "sensor_id": "lidar-demo",
    "sensor_health": "degraded", "position_m": [1737502, 1, 0], "position_std_m": [3, 3, 3],
})
estimate = StateSnapshot.model_validate(base | {"state_id": "estimate-demo", "position_std_m": [2, 2, 2]})
for snapshot in (truth, observation, estimate):
    print(snapshot.record_kind, snapshot.position_m, "propellant_kg:", snapshot.propellant_kg)
print("Event declaration:", scenarios["resource-outage"].events[0].payload.model_dump())
print("No event has been applied to a simulation.")

ground_truth (1737500.0, 0.0, 0.0) propellant_kg: 50.0
observation (1737502.0, 1.0, 0.0) propellant_kg: None
estimate (1737500.0, 0.0, 0.0) propellant_kg: None
Event declaration: {'kind': 'resource_outage', 'resource_id': 'shared-antenna', 'window': {'start_utc': datetime.datetime(2026, 11, 4, 0, 0, tzinfo=TzInfo(0)), 'end_utc': datetime.datetime(2026, 11, 4, 1, 0, tzinfo=TzInfo(0))}}
No event has been applied to a simulation.


## 7. A plan carries input identity, not an automatic feasibility claim

This manually authored selection is only a proposal. H0 checks IDs and the input snapshot. H1 will independently evaluate exactly-one selection, capacity, deadlines, objective values and missing inputs. Stale-input rejection here is an identity check; an asynchronous recovery controller belongs to H6.

In [8]:
proposal = Plan(
    plan_id="manual-example", scenario_id=scenario.scenario_id,
    scenario_revision=scenario.revision, input_fingerprint=fingerprint(scenario),
    state_revisions={state.state_id: state.revision for state in scenario.states},
    selected_option_ids=("mission-1-option-1", "mission-2-option-2", "mission-3-option-3"),
    execution_mode="classical", solver_name="manual-example-no-solver",
)
validate_plan_references(proposal, scenario)
assert proposal.validation_status == "unvalidated"
print("Reference check passed; physical feasibility remains", proposal.validation_status)
try:
    validate_plan_references(proposal, changed)
except ValueError as error:
    print("Changed input detected:", error)
else:
    raise AssertionError("Changed input was not detected")

Reference check passed; physical feasibility remains unvalidated
Changed input detected: Plan input identity does not match scenario


## H0 complete — next is H1

We now have shared contracts, traceable inputs, reproducible documents and illustrative benchmark cases. Next, H1 will enumerate the small problem and independently validate every proposed plan. H2 will encode that same problem as QUBO and run a QAOA experiment. **No quantum advantage or operational landing capability is demonstrated by this notebook.**